# Phase 251 — COACH420-clean scale-3 training list with true ligand centers

Phase 53/56 trained on the 7,513-chain round-3 pool after excluding every chain homologous (>=30% identity, MMseqs2)
to COACH420 (7,181 chains kept), with each chain's true ligand center recovered from its structure. This notebook reruns
exactly those steps (Phase 53's cells, verbatim) and saves the result -- the ordered key list and the true centers -- so the
geometry-control training kernels all start from one fixed, identical training set instead of each redoing the MMseqs2 run.

## Cell 1 — Install

In [ ]:
!pip install -q torch-geometric==2.6.1 gudhi==3.10.1 biopython==1.84 transformers==4.46.3

## Cell 2 — Setup (Phase 53 cell; the unused Phase 51 topology blob is not loaded)

In [ ]:
import csv as _csv
import gzip, json, os, pickle, shutil, subprocess, time, urllib.request, warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from scipy.spatial import cKDTree
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

warnings.filterwarnings("ignore")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob10 = torch.load(locate("phase52_graphs.pt"), map_location="cpu", weights_only=False)
dataset = blob10["dataset"]
N_PRE = int(blob10["n_pre_topo"]); TOPO_DIM = int(blob10["topo_dim"])
POS_WEIGHT = float(blob10["pos_weight"])

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
print(f"training set: {len(dataset):,} chains")

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

## Cell 3 — Fetch ALL 420 COACH420 structures (Phase 53, verbatim)

In [ ]:
def parse_ds(url):
    lines = urllib.request.urlopen(url, timeout=60).read().decode().splitlines()
    out = []
    for ln in lines:
        ln = ln.strip()
        if not ln or ln.startswith("#"): continue
        name = ln.split("/")[-1].replace(".pdb", "")
        out.append((name[:4].upper(), name[4:].upper() or None))
    return out

coach = parse_ds("https://raw.githubusercontent.com/rdk/p2rank-datasets/master/coach420.ds")
print(f"COACH420: {len(coach)} entries (full benchmark, no filtering)")

need = sorted({pid for pid, _ in coach})
def fetch(pid):
    for ext in ("pdb","cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("pdb","cif"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=24) as ex: got = list(ex.map(fetch, need))
coach_paths = {p: f for p, f in got if f}
print(f"fetched {len(coach_paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

## Cell 4 — Clean the TRAINING side: exclude any chain homologous to COACH420 (Phase 53, verbatim)

In [ ]:
from Bio.PDB import PDBParser

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}

def get_seq(pid, path, chain_id):
    try:
        with gzip.open(path, "rt") as fh:
            model = next(PDBParser(QUIET=True, PERMISSIVE=True).get_structure(pid, fh).get_models())
        ch = model[chain_id] if chain_id else next(iter(model))
        residues = [r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
        return "".join(AA3_TO_1.get(r.get_resname().strip().upper(),"X") for r in residues)
    except Exception:
        return None

coach_seqs = {}
for pid, ch in coach:
    if pid not in coach_paths: continue
    seq = get_seq(pid, coach_paths[pid], ch)
    if seq: coach_seqs[f"{pid}_{ch or 'A'}"] = seq
print(f"COACH420 sequences recovered: {len(coach_seqs)}/{len(coach)}")

mm = None
try:
    subprocess.run("curl -sL https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz -o /tmp/mm.tar.gz "
                   "&& tar xzf /tmp/mm.tar.gz -C /tmp", shell=True, check=True, timeout=600)
    c = Path("/tmp/mmseqs/bin/mmseqs")
    if c.exists(): os.chmod(c, 0o755); mm = str(c)
except Exception as e:
    print("mmseqs2 unavailable:", type(e).__name__)

W = Path("/tmp/rev_leak"); shutil.rmtree(W, ignore_errors=True); W.mkdir()
fasta_lines = [f">TRAIN__{d.pdb_id}_{d.chain_id}\n{d.seq}\n" for d in dataset]
fasta_lines += [f">BENCH__{k}\n{v}\n" for k, v in coach_seqs.items()]
(W/"all.fasta").write_text("".join(fasta_lines))

subprocess.run([mm, "easy-cluster", str(W/"all.fasta"), str(W/"res"), str(W/"tmp"),
                "--min-seq-id", "0.30", "-c", "0.8", "--cov-mode", "1", "-v", "1"],
               check=True, capture_output=True, timeout=3600)
rep = defaultdict(list)
for line in (W/"res_cluster.tsv").read_text().splitlines():
    a, b = line.split("\t"); rep[a].append(b)
for r_, members in rep.items():
    if r_ not in members: members.append(r_)

leaked_train_keys = set()
for members in rep.values():
    has_train = any(m.startswith("TRAIN__") for m in members)
    has_bench = any(m.startswith("BENCH__") for m in members)
    if has_train and has_bench:
        leaked_train_keys.update(m for m in members if m.startswith("TRAIN__"))

print(f"training chains homologous to COACH420 (excluded): {len(leaked_train_keys)}/{len(dataset)}")

clean_dataset = [d for d in dataset if f"TRAIN__{d.pdb_id}_{d.chain_id}" not in leaked_train_keys]
print(f"clean training set: {len(clean_dataset):,}/{len(dataset):,} chains "
      f"({100*len(clean_dataset)/len(dataset):.1f}% retained)")
assert len(clean_dataset) > 0.8 * len(dataset), "too much training data excluded -- investigate before proceeding"
dataset = clean_dataset

## Cell 5 — Recover true ligand centers for the cleaned training set (Phase 53, verbatim)

In [ ]:
from Bio.PDB import PDBParser

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}
NON_LIGAND={"HOH","DOD","WAT","SO4","PO4","GOL","EDO","PEG","PG4","PGE","1PE","MPD","TRS",
            "ACT","ACY","FMT","CIT","MES","EPE","DMS","IMD","BME","NO3","AZI","CO3","IOD",
            "BR","CL","F","NA","K","CS","RB","LI","UNX","UNL","UNK"}
CATALYTIC_METALS={"ZN","MG","MN","FE","FE2","CU","CU1","CA","NI","CO","CD","W","MO"}
MIN_LIG_ATOMS = 6
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

need = sorted({d.pdb_id for d in dataset})
def fetch(pid):
    for ext in ("pdb","cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("pdb","cif"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=16) as ex: got = list(ex.map(fetch, need))
paths = {p: f for p, f in got if f}
print(f"fetched {len(paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

def ligands_and_ca(pid, path, chain_id, n_expect):
    from Bio.PDB import MMCIFParser
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path,"rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
        ch = model[chain_id]
    except Exception:
        return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if len(residues) != n_expect: return None
    raw_ca = np.asarray([r["CA"].coord for r in residues], dtype=np.float32)

    ligs = []
    for c2 in model:
        for res in c2:
            if res.id[0] == " ": continue
            nm = res.get_resname().strip().upper()
            if nm in AA3_TO_1 or nm in NON_LIGAND: continue
            xyz = np.asarray([a.coord for a in res.get_unpacked_list() if a.element not in ("H","D")],
                             dtype=np.float32).reshape(-1,3)
            is_metal = nm in CATALYTIC_METALS
            if is_metal or len(xyz) >= MIN_LIG_ATOMS: ligs.append(xyz)
    return raw_ca, ligs

t0=time.time(); n_ok=0; n_geom_fail=0; n_no_ligand=0
for d in dataset:
    if d.pdb_id not in paths: continue
    r = ligands_and_ca(d.pdb_id, paths[d.pdb_id], d.chain_id, int(d.num_nodes))
    if r is None: continue
    raw_ca, ligs = r
    if not ligs: n_no_ligand += 1; continue

    # geometry-drift check: pairwise CA distances must match d.pos (residue ORDER, not just count)
    k = min(50, len(raw_ca))
    a = np.linalg.norm(raw_ca[:k,None]-raw_ca[None,:k], axis=-1)
    b = np.linalg.norm(d.pos.numpy()[:k,None]-d.pos.numpy()[None,:k], axis=-1)
    if float(np.abs(a-b).max()) > 1e-2:
        n_geom_fail += 1; continue

    offset = raw_ca.mean(axis=0)                       # same centering d.pos already used
    y_np = d.y.numpy().astype(bool)
    contact_centroid = raw_ca[y_np].mean(axis=0) if y_np.any() else raw_ca.mean(axis=0)
    # "primary" ligand = the one whose center of mass is nearest the labelled contact residues
    coms = [lig.mean(axis=0) for lig in ligs]
    best = int(np.argmin([np.linalg.norm(c - contact_centroid) for c in coms]))
    d.true_center = torch.from_numpy((coms[best] - offset).astype(np.float32))
    n_ok += 1

print(f"\ntrue centers recovered: {n_ok:,}/{len(dataset):,} "
      f"({(time.time()-t0)/60:.1f} min) | geometry mismatches: {n_geom_fail} | no ligand: {n_no_ligand}")

dataset = [d for d in dataset if hasattr(d, "true_center")]
print(f"clean training chains with recovered true centers: {len(dataset):,}")

rng = np.random.RandomState(42)
perm = rng.permutation(len(dataset))
n_val = max(1, int(0.08 * len(perm)))
splits = {"val": [dataset[i] for i in perm[:n_val]], "train": [dataset[i] for i in perm[n_val:]]}
n_tr, n_va = len(splits["train"]), len(splits["val"])
print(f"deployment training: {n_tr} train / {n_va} val")

## Cell 6 — Save the fixed training list and centers

In [ ]:
order_keys = [(str(d.pdb_id), str(d.chain_id)) for d in dataset]          # dataset order after all filters
centers = {(str(d.pdb_id), str(d.chain_id)): d.true_center.numpy().astype(np.float32) for d in dataset}
assert len(order_keys) == len(set(order_keys)) == len(centers)
torch.save(dict(order_keys=order_keys, centers=centers, n_train_excluded=len(leaked_train_keys),
                n_clean=len(order_keys)), WORK / "phase251_clean_centers.pt")
print(f"saved {len(order_keys):,} chains with true centers -> phase251_clean_centers.pt "
      f"({len(leaked_train_keys)} excluded as homologous to COACH420)")
assert 7000 <= len(order_keys) <= 7300, f"expected about 7,181 chains (Phase 53), got {len(order_keys)}"